# CANFAR Rubin Data Access and INJECT

A step-by-step path from a CANFAR Rubin notebook session to a small injection/recovery test on a real coadd. Start with DP2; change `DATASET` to `dp1` to try DP1. No collection, tract, or patch is assumed.

Open this notebook in your **CANFAR Rubin Science Notebook** session, from the checkout at `/arc/home/snair25/INJECT`. Select the Rubin Science Pipelines kernel and run sections in order. Pause at collection selection to choose from the live listing.

The session is expected to load `/opt/lsst/software/stack`, configure CADC credentials, and register Butler aliases at startup. This notebook does not install Rubin or change the server configuration. The optional token prompt is hidden: enter secrets only there, never in notebook source, chat, or a terminal command.

## 1. Check the notebook kernel
The executable is printed before importing Rubin, so it remains visible if an import fails. A missing `lsst` import is an environment problem, not a data-access or token problem.

In [ ]:
from pathlib import Path
import sys

print('Python:', sys.executable)
print('Working directory:', Path.cwd())
try:
    from lsst.daf.butler import Butler
    import lsst.afw.image
except ImportError as exc:
    raise ImportError(
        'This kernel could not import the Rubin stack. Select the Rubin Science '
        'Pipelines kernel in CANFAR JupyterLab and restart. Do not pip-install '
        'Rubin into a plain Python environment. If the correct kernel still fails, '
        'check the original exception below with CANFAR support.'
    ) from exc
print('Rubin imports OK')

## 2. Choose the dataset and inspect session credentials
CANFAR `dp1`/`dp2` use **CADC/CANFAR** credentials. Rubin-hosted `data.lsst.cloud` endpoints require a separate Rubin token. `dp02` is DP0.2, not DP2.

Only credential **presence** is displayed below, never values. Presence does not prove validity or access permission. Butler aliases and `RSPDiscovery` labels are separate registries; this notebook uses Butler directly.

Leave `USE_TOKEN_PROMPT=False` to use the session's configured authentication. Set it to `True` only to enter a CADC token in a hidden runtime prompt. Paste the token itself, without a `Bearer ` prefix. The token stays in kernel memory and is not written by this notebook. Restart the kernel to discard credentials held by Butler objects.

In [ ]:
import os

DATASET = 'dp2'
USE_TOKEN_PROMPT = False
N_CLUSTERS = 25
CUTOUT_SIZE = 800
SEED = 42

CANFAR_REPOS = {
    'dp1': 'https://ws-uv.canfar.net/lsst/api/butler/repo/dp1/butler.yaml',
    'dp2': 'https://ws-uv.canfar.net/lsst/api/butler/repo/dp2/butler.yaml',
}
if DATASET not in CANFAR_REPOS:
    raise ValueError('Choose dp1 or dp2 for this CANFAR-only workflow.')
REPO_URL = CANFAR_REPOS[DATASET]
for name in ('NUBLADO_TOKEN', 'ACCESS_TOKEN', 'CADC_TOKEN'):
    print(f'{name} present: {bool(os.environ.get(name))}')
print('Session token file present:', Path('/etc/nublado/secrets/token').is_file())
print('Butler repository index configured:', bool(os.environ.get('DAF_BUTLER_REPOSITORY_INDEX')))
print('Target:', REPO_URL)

## 3. Connect to Butler
The explicit CANFAR URL avoids dependence on aliases; aliases are listed for reference. Authentication is not silently switched to another service if a request fails.

The optional prompt uses `getpass` and `RemoteButlerFactory.create_butler_for_access_token`. If your installed stack lacks that API, use session authentication or ask CANFAR support about the image version. Do not substitute an unverified environment variable.

In [ ]:
from getpass import getpass, GetPassWarning
import warnings

print('Known Butler aliases:', sorted(Butler.get_known_repos()))
if USE_TOKEN_PROMPT:
    from lsst.daf.butler.remote_butler import RemoteButlerFactory

    factory = RemoteButlerFactory.create_factory_for_url(REPO_URL)
    with warnings.catch_warnings():
        warnings.simplefilter('error', GetPassWarning)
        access_token = getpass('CADC/CANFAR token (hidden): ').strip()
    try:
        if not access_token or access_token.lower().startswith('bearer '):
            raise ValueError('Enter the token itself, without a Bearer prefix.')
        butler = factory.create_butler_for_access_token(access_token)
    finally:
        del access_token
else:
    butler = Butler(REPO_URL)
print('Butler constructed; the next queries check service access.')

## 4. Discover collections and coadd dataset types
This makes real metadata requests. A 401/403 here points to authentication or authorization, not INJECT. Refresh your CANFAR session/certificate or use the hidden CADC token prompt, then rerun the connection cell. Do not share token values.

DP2 lists the science exposure as **`deep_coadd`**, while older repositories may use **`deepCoadd`**. The next cell selects one of these exact names from the live listing. It never substitutes summary tables, maps, or pretty-picture products. Review the printed storage class and dimensions; section 6 also checks that the retrieved object has the image, mask, bounding box, and PSF interface required by INJECT. Set `COADD_DATASET_TYPE` explicitly below if both names exist and you need the legacy product.

In [ ]:
collections = sorted(butler.collections.query('*'))
print(f'Available collections ({len(collections)}):')
for collection_name in collections:
    print(' ', collection_name)
dataset_types = {item.name: item for item in butler.registry.queryDatasetTypes('*')}
print()
print('Coadd-related dataset types:')
for dataset_type_name in sorted(dataset_types):
    if 'coadd' in dataset_type_name.lower():
        print(' ', dataset_type_name)

COADD_DATASET_TYPE = None
if COADD_DATASET_TYPE is None:
    COADD_DATASET_TYPE = next(
        (name for name in ('deep_coadd', 'deepCoadd') if name in dataset_types), None
    )
if COADD_DATASET_TYPE not in dataset_types:
    raise RuntimeError('No selected science coadd type is registered. Inspect the live dataset listing.')
coadd_type = dataset_types[COADD_DATASET_TYPE]
print('Selected dataset type:', COADD_DATASET_TYPE)
print('Storage class:', coadd_type.storageClass_name)
print('Dimensions:', sorted(coadd_type.dimensions.names))
if not {'tract', 'patch', 'band'}.issubset(coadd_type.dimensions.names):
    raise RuntimeError('Selected product does not have the expected tract/patch/band coadd dimensions.')

## 5. Select a collection and find real images
The collection matching `DATASET` (for example, `dp2`) is suggested only if it exists in the live collection list. You can replace it with another listed science collection. This is a starting point, not a guarantee of matching images. Begin with band `i`; try another available band if there are no results. The query returns at most five candidates rather than scanning all image data. Their ordering is not a scientific selection.

No DP0.2 collection or tract/patch placeholder is used. Retain the full returned data ID, including `skymap` if present.

In [ ]:
COLLECTION = DATASET if DATASET in collections else None
BAND = 'i'

if not COLLECTION:
    raise ValueError('Set COLLECTION to a collection printed in section 4, then rerun this cell.')
candidates = list(butler.query_datasets(
    COADD_DATASET_TYPE, collections=[COLLECTION],
    where='band = :band', bind={'band': BAND},
    find_first=True, limit=5,
))
if not candidates:
    raise RuntimeError(f'No {COADD_DATASET_TYPE} candidates found. Try another collection or band.')
print('Queried collection:', COLLECTION)
for index, candidate in enumerate(candidates):
    print(index, 'run:', candidate.run, 'data ID:', dict(candidate.dataId.required))

## 6. Fetch and inspect one coadd
Choose a candidate index. Pinning to its run ensures the preview and INJECT load use the same product. Fetching pixels is a separate access check from listing metadata.

DP2 can return `lsst.images.cells.CellCoadd`, which uses **`to_legacy()`** to create a Rubin exposure. Older `lsst.cell_coadds.MultipleCellCoadd` products use `stitch().asExposure()`; already stitched products only need `asExposure()`. The preview prints the raw Python type and supports each path. The INJECT loader applies the same conversion. This preserves the exposure view's image, mask, PSF, and coordinate origin rather than passing only a NumPy image.

The modern `to_legacy()` API also sets WCS and photometric calibration but does not retain every aperture-correction/provenance component in the legacy exposure. Its documented metadata requirements still apply; conversion errors are not suppressed. See the [CellCoadd source](https://github.com/lsst/images/blob/main/python/lsst/images/cells/_coadd.py).

**Science caveat:** converting a cell coadd does not smooth the data or PSF across cell boundaries. This smoke test is not a validation of cell-boundary completeness.

**Resource note:** both this preview and the current INJECT loader fetch a full coadd before selecting its lower-left `CUTOUT_SIZE` square. Conversion can materialize full image planes. An 800-pixel cutout does not bound download size or peak memory. INJECT will fetch again after the preview is released. Choose another candidate if the cutout has no usable science pixels or PSF.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

CANDIDATE_INDEX = 0
if not 0 <= CANDIDATE_INDEX < len(candidates):
    raise ValueError('Choose one of the candidate indices printed above.')
selected = candidates[CANDIDATE_INDEX]
DATA_ID = dict(selected.dataId.required)
BUTLER_COLLECTIONS = selected.run
science_butler = butler.clone(collections=[BUTLER_COLLECTIONS])
print('Dataset type:', COADD_DATASET_TYPE)
print('Run:', BUTLER_COLLECTIONS)
print('Data ID:', DATA_ID)
exposure = science_butler.get(COADD_DATASET_TYPE, dataId=DATA_ID)
try:
    print('Raw product type:', f'{type(exposure).__module__}.{type(exposure).__qualname__}')
    if callable(getattr(exposure, 'to_legacy', None)):
        exposure = exposure.to_legacy()
    if callable(getattr(exposure, 'stitch', None)):
        exposure = exposure.stitch()
    if callable(getattr(exposure, 'asExposure', None)):
        exposure = exposure.asExposure()
    print('Exposure view type:', f'{type(exposure).__module__}.{type(exposure).__qualname__}')
    required_attributes = ('image', 'mask', 'getPsf', 'getBBox')
    if any(not hasattr(exposure, name) for name in required_attributes):
        raise TypeError(
            f'Unsupported product type {type(exposure).__module__}.{type(exposure).__qualname__}: '
            'expected an exposure, CellCoadd.to_legacy(), or stitch().asExposure().'
        )
    if (not hasattr(exposure.image, 'array') or not hasattr(exposure.mask, 'array')
            or not callable(getattr(exposure.mask, 'getMaskPlaneDict', None))):
        raise TypeError('Selected product does not expose the image/mask arrays required by INJECT.')
    print('Full image shape:', exposure.image.array.shape)
    print('Bounding box:', exposure.getBBox())
    print('PSF type:', type(exposure.getPsf()).__name__)
    if exposure.getPsf() is None:
        raise RuntimeError('No PSF available; select another candidate.')
    preview = exposure.image.array[:CUTOUT_SIZE, :CUTOUT_SIZE].copy()
    finite = preview[np.isfinite(preview)]
    if finite.size == 0:
        raise RuntimeError('No finite pixels in the cutout; select another candidate.')
    vmin, vmax = np.percentile(finite, [1, 99])
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(preview, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
    ax.set_title(f"{DATASET.upper()} {DATA_ID['band']}-band science cutout")
    ax.set_xlabel('Cutout x [pixels]')
    ax.set_ylabel('Cutout y [pixels]')
    plt.show()
finally:
    del exposure
print('Real image access succeeded. Continue to INJECT below.')

## 7. Import INJECT from your checkout
The data-access steps above do not depend on INJECT. Now locate your checkout and verify that Python imports this project's `inject`, not another package with the same name.

If an INJECT dependency is missing, install this checkout into the **same Rubin-capable kernel** with IPython's `%pip install -e /arc/home/snair25/INJECT`, then restart the kernel and rerun. Installation is not automatic here. The plain CANFAR venv setup does not by itself provide the Rubin stack. If the wrong `inject` is already cached, restart the kernel instead of trying to reload it.

In [ ]:
search_paths = [Path.cwd(), *Path.cwd().parents, Path('/arc/home/snair25/INJECT')]
REPO_ROOT = next((
    path.resolve() for path in search_paths
    if (path / 'inject' / '__init__.py').is_file() and (path / 'pyproject.toml').is_file()
), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Open this notebook from your INJECT checkout or update search_paths.')
sys.path.insert(0, str(REPO_ROOT))
import inject

package_file = getattr(inject, '__file__', None)
required = ('ClusterConfig', 'InjectionConfig', 'InjectionPipeline', 'notebook_output_dir')
if (package_file is None
        or Path(package_file).resolve() != REPO_ROOT / 'inject' / '__init__.py'
        or any(not hasattr(inject, name) for name in required)):
    raise ImportError('Wrong inject package loaded. Restart the kernel and use this checkout.')
from inject import ClusterConfig, InjectionConfig, InjectionPipeline, notebook_output_dir
from inject.detection import run_cluster_detection
print('INJECT imported from:', package_file)

## 8. Configure and load a small injection run
Use 25 smooth King-profile clusters and the actual Rubin PSF. Mask flags are recorded, but flagged locations are not automatically excluded in this initial diagnostic run. Positions and sizes are in pixels. Outputs are local files in the checkout's `plots` directory; the remote Butler repository is not modified.

In [ ]:
RUN_OUTPUT_DIR = notebook_output_dir(f'canfar_{DATASET}_smoke_test')
PLOT_OUTPUT_DIR = RUN_OUTPUT_DIR / 'plots'
config = InjectionConfig(
    run_name=f'canfar_{DATASET}_smoke_test',
    band=DATA_ID['band'], cutout_size=CUTOUT_SIZE,
    n_clusters=N_CLUSTERS, seed=SEED, edge_buffer=50,
    use_actual_psf=True, psf_fwhm_fallback=3.5,
    record_psf_mask_flags=True, skip_bad_psf_regions=False,
    save_injected_image=False, output_dir=str(RUN_OUTPUT_DIR),
    tract=DATA_ID.get('tract'), patch=DATA_ID.get('patch'),
    cluster_config=ClusterConfig(
        profile_type='king', method='smooth',
        mag_min=20.5, mag_max=24.5, r_half_min=2.0, r_half_max=8.0,
        concentration_min=5.0, concentration_max=20.0,
    ),
)
pipe = InjectionPipeline(config)
pipe.load_data(butler=science_butler, data_id=DATA_ID, dataset_type=COADD_DATASET_TYPE)
finite = pipe.image[np.isfinite(pipe.image)]
if finite.size == 0:
    raise RuntimeError('Loaded cutout has no finite science pixels.')
vmin, vmax = np.percentile(finite, [1, 99])
print('Loaded cutout:', pipe.image.shape)
print('Output directory:', RUN_OUTPUT_DIR)

## 9. Inject and inspect the difference
The difference panel should show positive residuals at injected positions. Green circles mark up to ten injections in the middle panel. A successful injection does not yet establish detection completeness.

In [ ]:
catalog = pipe.generate_catalog()
injected_image, injection_info = pipe.inject(catalog=catalog, drop_stamps=True, verbose=True)
print('Generated:', len(catalog), 'Kept:', len(injection_info))
if not injection_info:
    raise RuntimeError('No injections were kept. Inspect PSF/mask messages or select another candidate.')
difference = injected_image.astype(float) - pipe.image.astype(float)
finite_difference = np.abs(difference[np.isfinite(difference)])
if finite_difference.size == 0:
    raise RuntimeError('Difference image has no finite pixels.')
difference_limit = max(float(np.percentile(finite_difference, 99.5)), 1e-6)
fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
axes[0].imshow(pipe.image, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
axes[1].imshow(injected_image, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
axes[2].imshow(difference, origin='lower', cmap='RdBu_r',
               vmin=-difference_limit, vmax=difference_limit)
for axis, title in zip(axes, ('Original', 'Injected', 'Difference')):
    axis.set_title(title)
    axis.set_xticks([])
    axis.set_yticks([])
for entry in injection_info[:10]:
    axes[1].plot(entry['x'], entry['y'], 'o', ms=5, mec='lime', mfc='none', mew=0.9)
plt.show()

## 10. Run a quick detection and recovery check
The built-in matched-filter/MCI detector is a diagnostic, not a calibrated science detector. Some injections may be too faint to recover. Zero recoveries is not automatically an access or injection failure; inspect the images and detector settings before drawing conclusions.

In [ ]:
psf_samples = [
    float(row['psf_fwhm_px']) for row in injection_info
    if row.get('psf_fwhm_px') is not None
    and np.isfinite(row['psf_fwhm_px']) and row['psf_fwhm_px'] > 0
]
detector_psf_fwhm = float(np.median(psf_samples)) if psf_samples else config.psf_fwhm_fallback
detections = pipe.detect_with(
    run_cluster_detection, psf_fwhm=detector_psf_fwhm,
    threshold_sigma=3.0, mci_max=0.85, snr_min=3.0,
    r_half_min=1.0, ellipticity_max=0.6, box_size=64,
    pixel_scale=config.pixel_scale, use_multiscale=True, use_mci=True, verbose=False,
)
stats = pipe.analyze(match_radius=5.0)
stats

## 11. Save results and dataset provenance
Save catalogs, diagnostic plots, and a small provenance record containing only the dataset selection and run settings. No token, certificate, or environment dump is saved. Rerunning the same dataset/run name may overwrite outputs; change `run_name` and the output directory for comparisons.

In [ ]:
import json

PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pipe.save_results()
plot_results = pipe.make_plots(
    output_dir=str(PLOT_OUTPUT_DIR),
    plots=['injection_summary', 'before_after', 'position_map', 'completeness_1d', 'psf_fwhm_hist'],
    show=True, save=True, n_stamps=min(6, len(pipe.injection_info)),
)
provenance = {
    'dataset': DATASET, 'repo': REPO_URL, 'collection': BUTLER_COLLECTIONS,
    'dataset_type': COADD_DATASET_TYPE,
    'data_id': DATA_ID, 'dataset_id': str(selected.id),
    'n_clusters': N_CLUSTERS, 'cutout_size': CUTOUT_SIZE, 'seed': SEED,
}
with (RUN_OUTPUT_DIR / 'canfar_access_provenance.json').open('w') as handle:
    json.dump(provenance, handle, indent=2, default=str)
print('Results:', RUN_OUTPUT_DIR)
for name, path in plot_results['saved'].items():
    print(f'{name}: {path}')

## Troubleshooting

| Symptom | Next check |
| --- | --- |
| `No module named lsst` or a Rubin dependency import failure | Confirm the selected kernel is the CANFAR Rubin kernel. A token cannot fix an import failure. |
| Wrong `inject` package or missing INJECT dependency | Install the checkout in the same kernel environment, restart, and rerun. |
| `load_data` rejects `dataset_type` | Update the CANFAR checkout's `inject/pipeline.py` as well as this notebook, then restart the kernel. |
| 401/403 during collection queries or pixel retrieval | Check CADC credential validity, permissions, and the CANFAR endpoint. Refresh the session or use the hidden prompt. Never share credentials. |
| Hidden input unavailable / `GetPassWarning` | Stop; do not enter a token into echoed input. Use a notebook frontend supporting hidden input or session authentication. |
| `RemoteButlerFactory` API missing | Use session authentication or ask support about the installed stack version. |
| Collection or band has no candidates | Choose another collection/band from the repository; do not reuse old DP0.2 data IDs. |
| Neither science coadd name is registered | Inspect the live listing; do not substitute summary tables, maps, or pretty pictures. |
| Product lacks image/mask/PSF/bounding box | Stop before INJECT; the selected product is not a compatible science exposure. |
| Missing or inconsistent dimensions | Keep the entire returned `dataId.required`, including `skymap` where required. |
| Invalid PSF or empty/all-masked cutout | Select another candidate and rerun sections 6 onward; inspect mask/PSF diagnostics. |
| No recovered injections | Inspect injection residuals and detector settings; this is not by itself an authentication failure. |

After token entry, avoid dumping `os.environ`, Butler object internals, request headers, or debug HTTP logs. Restart the kernel when finished. If sharing an error, inspect and redact it first. CANFAR support: **support@canfar.net**.